# Cost vs. Performance Tradeoff

Plots `(1 - Brier score)` versus average output tokens spent by each confidence estimator on a given (model, dataset) pair, so all four methods can be compared head-to-head with the random baseline (analytic) and the model's own answer-generation cost as reference lines.

Default cell below produces the `cc` figure for `Qwen3-8B-non-thinking` on `simpleqa-verified`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path('..').resolve()
OUTPUTS_DIR = REPO_ROOT / 'outputs'
EST_DIR = REPO_ROOT / 'estimator_results'

In [ ]:
# ===== CONFIGURATION =====
MODEL = 'Qwen3-8B-non-thinking'
DATASET = 'simpleqa-verified'
CALIB = 'cc'           # cc | rc | rc-input_cc-target  (only used for probing & verbalized/ptrue file names)
K_CONSISTENCY = 5      # which k of the consistency estimator to plot

# Map an eval dataset -> the linear-probe training split we treat as 'self-trained'.
# (Used to pick the probing point: probe trained on the same task's train split.)
EVAL_TO_TRAIN = {
    'triviaqa-validation': 'triviaqa-train',
    'simpleqa-verified':   'simpleqa-train',
    'gsm8k-test':          'gsm8k-train',
    'math-500':            'math-train',
    'aime-test':           'aime-train',
    'aime25-test':         'aime-train',
    'mmlu-validation':     'mmlu-validation',
    'mmlu-test':           'mmlu-validation',
    'gpqa-diamond':        'gpqa-train',
}

PRETTY_DATASET = {
    'triviaqa-validation': 'TriviaQA',
    'simpleqa-verified':   'SimpleQA',
    'gsm8k-test':          'GSM8K',
    'math-500':            'MATH-500',
    'aime-test':           'AIME',
    'mmlu-test':           'MMLU',
    'gpqa-diamond':        'GPQA',
}
PRETTY_MODEL = {
    'Qwen3-8B-non-thinking': 'Qwen3-8B',
    'Olmo-3-7B-Instruct':    'Olmo-3-7B-Instruct',
    'gpt-oss-20b':           'gpt-oss-20b',
}

In [ ]:
def jsonl_iter(path: Path):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def avg_completion_tokens(path: Path) -> float:
    tot, n = 0, 0
    for r in jsonl_iter(path):
        if 'completion_tokens' in r:
            tot += r['completion_tokens']
            n += 1
    if n == 0:
        raise RuntimeError(f'no completion_tokens in {path}')
    return tot / n


def load_json(path: Path) -> dict:
    with open(path) as f:
        return json.load(f)

In [ ]:
out_run = OUTPUTS_DIR / f'{DATASET}__{MODEL}'
sampled_path = out_run / 'sampled.jsonl'
ground_truth_path = out_run / 'ground_truth.jsonl'

# === Average answer-generation cost (the vertical reference line) ===
avg_answer_tokens = avg_completion_tokens(sampled_path)

# === Analytic uniform-random Brier baseline ===
# E[L] = (1/N) sum_i (1/3 - mu_i + mu_i^2)
mus = np.array([r['expected_accuracy'] for r in jsonl_iter(ground_truth_path)])
random_brier = float(np.mean(1.0 / 3.0 - mus + mus ** 2))
random_perf = 1.0 - random_brier

print(f'avg answer-generation tokens: {avg_answer_tokens:.1f}')
print(f'mean accuracy mu_bar       : {mus.mean():.4f}')
print(f'random Brier (analytic)    : {random_brier:.4f}  -> 1 - Brier = {random_perf:.4f}')

In [ ]:
def consistency_point(k: int):
    metrics = load_json(
        EST_DIR / 'consistency' / f'{DATASET}__{MODEL}' / f'consistency_k{k}_metrics.json'
    )
    # Cost: k independent samples of the same answer-generation prompt.
    return k * avg_answer_tokens, 1.0 - metrics['mse']


def verbalized_point():
    base = EST_DIR / 'verbalized_confidence' / f'{DATASET}__{MODEL}'
    metrics = load_json(base / f'evaluation_metrics_{CALIB}.json')
    tokens = avg_completion_tokens(base / f'confidence_predictions_{CALIB}.jsonl')
    return tokens, 1.0 - metrics['mse']


def ptrue_point():
    base = EST_DIR / 'ptrue' / f'{DATASET}__{MODEL}'
    metrics = load_json(base / f'evaluation_metrics_{CALIB}.json')
    tokens = avg_completion_tokens(base / f'confidence_predictions_{CALIB}.jsonl')
    return tokens, 1.0 - metrics['mse']


def probing_point():
    train_split = EVAL_TO_TRAIN[DATASET]
    metrics = load_json(
        EST_DIR / 'linear_probe'
        / f'{MODEL}__trained-on-{train_split}-{CALIB}'
        / DATASET / 'metrics.json'
    )
    # Probing reads a hidden state of the answer-generation forward pass; no extra generation.
    return 1.0, 1.0 - metrics['mse']


points = {
    f'Consistency ($k_c={K_CONSISTENCY}$)': consistency_point(K_CONSISTENCY),
    'Verbalized':                            verbalized_point(),
    'P(True)':                               ptrue_point(),
    'Probing':                               probing_point(),
}
for name, (x, y) in points.items():
    print(f'{name:<28s}  tokens={x:8.2f}   1-Brier={y:.4f}')

In [ ]:
STYLE = {
    f'Consistency ($k_c={K_CONSISTENCY}$)': dict(marker='o', color='#1f77b4', s=180),
    'Verbalized':                            dict(marker='s', color='#ff7f0e', s=180),
    'P(True)':                               dict(marker='^', color='#2ca02c', s=180),
    'Probing':                               dict(marker='D', color='#d62728', s=180),
}

fig, ax = plt.subplots(figsize=(6, 4.5))
fontsize = 13.5

for name, (x, y) in points.items():
    ax.scatter(x, y, label=name, linewidth=0.6, **STYLE[name])

ax.axhline(random_perf, color='dimgray', linestyle='--', linewidth=3, label='Random Baseline')
ax.axvline(avg_answer_tokens, color='dimgray', linestyle='-', linewidth=3, label='Avg Output Tokens')

ax.set_xscale('log')
ax.set_xlabel('Output Tokens', fontsize=fontsize)
ax.set_ylabel('1 - Brier Score', fontsize=fontsize)
ax.set_title(
    f'{PRETTY_MODEL.get(MODEL, MODEL)} on {PRETTY_DATASET.get(DATASET, DATASET)}',
    fontsize=fontsize,
    fontweight='bold'
)
ax.tick_params(axis='both', labelsize=fontsize)

all_y = [y for _, y in points.values()] + [random_perf]
ymin = max(0.0, min(all_y) - 0.08)
ymax = min(1.02, max(all_y) + 0.08)
ax.set_ylim(ymin, ymax)

all_x = [x for x, _ in points.values()] + [avg_answer_tokens]
ax.set_xlim(0.5, max(all_x) * 3)

ax.grid(True, linestyle='-', alpha=0.25)
ax.legend(
    loc='lower center',           # which corner of the legend box anchors to bbox
    bbox_to_anchor=(0.42, -0.015),    # (x, y) in axes fraction; 0.5 = horizontally centered
    fontsize=fontsize - 0.5,
    framealpha=0.95,
)
fig.tight_layout()

out_path = Path(f'cost_performance__{DATASET}__{MODEL}__{CALIB}.png')
fig.savefig(out_path, bbox_inches='tight', dpi=300)
print(f'saved -> {out_path.resolve()}')
plt.show()

In [ ]:
# ===== 3 (models) x 7 (datasets) cost-vs-performance grid =====

MODELS = {
    'Olmo-3-7B-Instruct':    'Olmo-3-7B',
    'Qwen3-8B-non-thinking': 'Qwen3-8B',
    'gpt-oss-20b':           'gpt-oss-20b',
}
DATASETS = {
    'triviaqa-validation': 'TriviaQA',
    'simpleqa-verified':   'SimpleQA',
    'gsm8k-test':          'GSM8K',
    'math-500':            'MATH',
    'aime-test':           'AIME',
    'mmlu-test':           'MMLU',
    'gpqa-diamond':        'GPQA',
}
CALIB = 'cc'
K_CONSISTENCY = 5

# --- collectors that reuse the helpers from earlier cells ---
def _consistency(model, dataset, sampled_avg, k=K_CONSISTENCY):
    p = EST_DIR / 'consistency' / f'{dataset}__{model}' / f'consistency_k{k}_metrics.json'
    return k * sampled_avg, 1.0 - load_json(p)['mse']

def _verbalized(model, dataset, isotonic=False):
    base = EST_DIR / 'verbalized_confidence' / f'{dataset}__{model}'
    suffix = '_isotonic' if isotonic else ''
    metrics = load_json(base / f'evaluation_metrics_{CALIB}{suffix}.json')
    tokens = avg_completion_tokens(base / f'confidence_predictions_{CALIB}.jsonl')
    return tokens, 1.0 - metrics['mse']

def _ptrue(model, dataset, isotonic=False):
    base = EST_DIR / 'ptrue' / f'{dataset}__{model}'
    suffix = '_isotonic' if isotonic else ''
    metrics = load_json(base / f'evaluation_metrics_{CALIB}{suffix}.json')
    tokens = avg_completion_tokens(base / f'confidence_predictions_{CALIB}.jsonl')
    if model != 'gpt-oss-20b':
        tokens = 1
    return tokens, 1.0 - metrics['mse']

def _probe(model, dataset):
    train_split = EVAL_TO_TRAIN[dataset]
    p = EST_DIR / 'linear_probe' / f'{model}__trained-on-{train_split}-{CALIB}' / dataset / 'metrics.json'
    return 1.0, 1.0 - load_json(p)['mse']

# Method styles (marker + color). Isotonic variants share the marker and use a darker hue.
METHOD_STYLE = {
    f'Consistency ($k_c={K_CONSISTENCY}$)': dict(marker='o', color='#1f77b4'),
    'Verbalized':                            dict(marker='s', color='#ffc100'),
    'Verbalized + isotonic':                 dict(marker='s', color='#ff7400'),  # orange-red
    'P(True)':                               dict(marker='^', color='#2ca02c'),
    'P(True) + isotonic':                    dict(marker='^', color='#0e4d0e'),  # darker green
    'Probing':                               dict(marker='D', color='#d62728'),
}

n_rows, n_cols = len(MODELS), len(DATASETS)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.4 * n_cols, 2.1 * n_rows), sharex=False, dpi=500)

for i, (model, model_pretty) in enumerate(MODELS.items()):
    for j, (dataset, ds_pretty) in enumerate(DATASETS.items()):
        ax = axes[i, j]

        # === Per-(model,dataset) data ===
        out_run = OUTPUTS_DIR / f'{dataset}__{model}'
        sampled_avg = avg_completion_tokens(out_run / 'sampled.jsonl')
        mus = np.array([r['expected_accuracy'] for r in jsonl_iter(out_run / 'ground_truth.jsonl')])
        random_perf = 1.0 - float(np.mean(1.0 / 3.0 - mus + mus ** 2))

        pts = {}
        try: pts[f'Consistency ($k_c={K_CONSISTENCY}$)'] = _consistency(model, dataset, sampled_avg)
        except FileNotFoundError: pass
        try: pts['Verbalized']            = _verbalized(model, dataset, isotonic=False)
        except FileNotFoundError: pass
        try: pts['Verbalized + isotonic'] = _verbalized(model, dataset, isotonic=True)
        except FileNotFoundError: pass
        try: pts['P(True)']               = _ptrue(model, dataset, isotonic=False)
        except FileNotFoundError: pass
        try: pts['P(True) + isotonic']    = _ptrue(model, dataset, isotonic=True)
        except FileNotFoundError: pass
        try: pts['Probing']               = _probe(model, dataset)
        except FileNotFoundError: pass

        for name, (x, y) in pts.items():
            ax.scatter(x, y, s=80,
                       label=name, **METHOD_STYLE[name])

        ax.axhline(random_perf,  color='dimgray', linestyle='--', linewidth=2)
        ax.axvline(sampled_avg,  color='dimgray', linestyle='-',  linewidth=2)

        ax.set_xscale('log')
        ax.set_xlim(0.5, 5e4)

        ys = [y for _, y in pts.values()] + [random_perf]
        lo, hi = min(ys), max(ys)
        pad = max(0.02, 0.08 * (hi - lo))
        ax.set_ylim(max(0.0, lo - pad), min(1.02, hi + pad))
        ax.set_yticks(np.round(np.linspace(max(0.0, lo - pad/2), min(1.0, hi + pad/2), 3), 2))

        ax.tick_params(axis='both', labelsize=8)
        ax.grid(True, which='major', linestyle='-', alpha=0.2)

        if i == 0:
            ax.set_title(ds_pretty, fontsize=14, fontweight='bold')
        if j == 0:
            ax.set_ylabel(model_pretty, fontsize=14, fontweight='bold')

# Single legend on top
handles_labels = {}
for ax in axes.flat:
    for h, l in zip(*ax.get_legend_handles_labels()):
        handles_labels.setdefault(l, h)
legend_order = [
    f'Consistency ($k_c={K_CONSISTENCY}$)',
    'Verbalized', 'Verbalized + isotonic',
    'P(True)',    'P(True) + isotonic',
    'Probing',
    'Random Baseline', 'Avg Output Tokens',
]
# Add the two reference-line proxies
from matplotlib.lines import Line2D
handles_labels['Random Baseline']   = Line2D([0],[0], color='dimgray', linestyle='--', linewidth=2)
handles_labels['Avg Output Tokens'] = Line2D([0],[0], color='dimgray', linestyle='-',  linewidth=2)

handles = [handles_labels[l] for l in legend_order if l in handles_labels]
labels  = [l                  for l in legend_order if l in handles_labels]
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.51, 1.02),
           ncol=len(labels), fontsize=12, frameon=True)

fig.tight_layout(rect=[0, 0, 1, 0.96])

out = Path(f'cost_performance_grid__{CALIB}.pdf')
fig.savefig(out, bbox_inches='tight', dpi=500)
print(f'saved -> {out.resolve()}')
plt.show()